In [1]:
%%capture
!pip install unsloth trl transformers datasets wandb

In [2]:
!git clone https://github.com/vedevpatel/Escalate.git
%cd Escalate

Cloning into 'Escalate'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 37 (delta 8), reused 22 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 12.69 KiB | 12.69 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Escalate


In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=2048,
    load_in_4bit=True, # qlora
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
)

==((====))==  Unsloth 2026.5.5: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [5]:
from datasets import load_dataset
ds = load_dataset("json", data_files="data/processed/sft_train.jsonl", split="train")
print(ds)
print(ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['task', 'waste_label', 'messages'],
    num_rows: 94
})
{'task': 'Compute the sum of all prime numbers less than 100.', 'waste_label': 'clean', 'messages': [{'role': 'system', 'content': 'You are a compute arbitration router. Given a task and context, decide the most token-efficient action.'}, {'role': 'user', 'content': 'Compute the sum of all prime numbers less than 100.'}, {'role': 'assistant', 'content': 'Thought: I need to find all prime numbers less than 100, then sum them. I\'ll use a function to check for primality and iterate through numbers.\n```python\ndef is_prime(n):\n    """Check if a number is prime."""\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n\nprimes_sum = sum(n for n in range(2, 100) if is_prime(n))\n```'}]}


In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

def formatting_func(example):
    text = ""
    messages = example["messages"]
    for msg in messages:
        if isinstance(msg, dict):
            role = msg["role"]
            content = msg["content"]
        else:
            role = msg[0]
            content = msg[1]

        if role == "system":
            text += f"<start_of_turn>system\n{content}<end_of_turn>\n"
        elif role == "user":
            text += f"<start_of_turn>user\n{content}<end_of_turn>\n"
        elif role == "assistant":
            text += f"<start_of_turn>model\n{content}<end_of_turn>\n"
    return [text]

def tokenize(example):
    texts = formatting_func(example)
    result = tokenizer.tokenizer(
        texts[0],
        truncation=True,
        max_length=2048,
        padding=False,
    )
    return result

tokenized_ds = ds.map(tokenize, remove_columns=ds.column_names)
print(tokenized_ds)
print(tokenized_ds[0]["input_ids"][:10])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer.tokenizer,
    train_dataset=tokenized_ds,
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        output_dir="outputs/sft-v1",
        report_to="wandb",
        logging_steps=5,
        save_strategy="epoch",
        remove_unused_columns=False,
    ),
)

trainer.train()

Map:   0%|          | 0/94 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 94
})
[236820, 3041, 236779, 1340, 236779, 887, 236813, 9731, 107, 3048]


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 94 | Num Epochs = 3 | Total steps = 36
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,020,544 of 8,003,176,992 (0.09% trained)


Step,Training Loss
5,0.711893
10,0.538474
15,0.495390
20,0.346979
25,0.384321
30,0.305835
35,0.288002


Unsloth: Restored added_tokens_decoder metadata in outputs/sft-v1/checkpoint-12/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/sft-v1/checkpoint-24/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/sft-v1/checkpoint-36/tokenizer_config.json.


TrainOutput(global_step=36, training_loss=0.4358888136015998, metrics={'train_runtime': 107.1985, 'train_samples_per_second': 2.631, 'train_steps_per_second': 0.336, 'total_flos': 3053447779488768.0, 'train_loss': 0.4358888136015998, 'epoch': 3.0})

In [21]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token)

model.push_to_hub("vedevpatel/escalate-router-sft-v1", token=token)
tokenizer.tokenizer.push_to_hub("vedevpatel/escalate-router-sft-v1", token=token)
print("Model saved to HuggingFace Hub")

Saved model to https://huggingface.co/vedevpatel/escalate-router-sft-v1


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpg5_df5g9/tokenizer_config.json.


Model saved to HuggingFace Hub
